In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:

import sys, os
import numpy as np
import torch
import lerobot
print(lerobot.__file__)

project_root = os.path.abspath(os.path.join(os.getcwd(),'..'))
print(project_root)
sys.path.append(project_root)

from src.utils.logger import get_logger
logger = get_logger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/user/projects/objectVLA/vlas/lerobot/src/lerobot/__init__.py
/home/user/projects/objectVLA


In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

In [ ]:
from lerobot.policies.pi0.modeling_pi0 import PI0Policy
from lerobot.policies.factory import make_pre_post_processors
model_id = "lerobot/pi0_libero_finetuned_v044"
dir = '../output/pi0liberofintune'

2026-04-10 13:49:13,595 - TensorFlow version 2.21.0 available.


In [ ]:
import os

pi0_path = os.path.abspath("../vlas/checkpoints/pi0_libero_finetuned_v044")
print(pi0_path)
# pi0_path = 'lerobot/pi0'
policy = PI0Policy.from_pretrained(pretrained_name_or_path=pi0_path,
                                   local_files_only=True,
                                   device_map={"": str(device)},
                                   torch_dtype=torch.float16,
                                    low_cpu_mem_usage=True,
                                   ).eval()

preprocess, postprocess = make_pre_post_processors(
    policy.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)


2026-04-10 13:49:21,000 - Loaded config: PI0Config(n_obs_steps=1, input_features={'observation.images.image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 256, 256)), 'observation.images.image2': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 256, 256)), 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(8,)), 'observation.images.empty_camera_0': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(7,))}, device='cuda', use_amp=False, use_peft=False, push_to_hub=True, repo_id='pepijn223/pi0_libero_finetuned_extra2', private=None, tags=None, license=None, pretrained_path=PosixPath('pepijn223/pi0_libero_finetuned_extra'), paligemma_variant='gemma_2b', action_expert_variant='gemma_300m', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, num_inference_steps=10, time_sampling_beta_alpha=1.5, t

/home/user/projects/objectVLA/vlas/checkpoints/pi0_libero_finetuned_v044
The PI0 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


KeyboardInterrupt: 

In [ ]:
from datasets import load_dataset
from src.utils.observation_utils import (
    DatasetBuildConfig,
    build_analysis_items_from_dataset,
)
 
from src.utils.pi0_attention_utils import AnalysisConfig, batch_analyze_pi0_attention

I0000 00:00:1775707326.765837  452582 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775707326.926164  452582 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775707327.960711  452582 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
ds = load_dataset("lerobot/libero_object_image", cache_dir="../mydatasets/amylingchen/libero")
train_ds = ds["train"]

dataset_cfg = DatasetBuildConfig(
    resize_size=(224, 224),
    image_key="observation.images.image",
    wrist_image_key="observation.images.wrist_image",
    output_base_image_key="observation.images.image",
    output_wrist_image_key="observation.images.image2",
    state_key="observation.state",
    # object_name="cream cheese",
    instruction_template="pick up the {object} and place it in the basket",
)
idxs = list(range(430,600))
print(f"Building analysis items for indices: {len(idxs)}")
object_name ="cream cheese"
print(f"Object names: {object_name}")
items = build_analysis_items_from_dataset(
    train_ds,
    indices=idxs,
    
    device=device,
    config=dataset_cfg,
    object_names= [object_name]*len(idxs),
    pair_mode="zip",
    metadata_builder=lambda sample, idx: {
        "task_index": sample.get("task_index"),
    },
)

attn_cfg = AnalysisConfig(
    layer_idx=-1,
    action_token_idx=-1,
    head_idx=None,
    alpha=0.3,
    top_k=5,
    save_overlay_figures=False,
    save_mean_summary_figure=True,
    save_cluster_summary_figures=False,
)



: 

In [ ]:
output_dir=f"../output/pi0_attention_episode_2_{object_name}"
df = batch_analyze_pi0_attention(
    items,
    policy=policy,
    preprocess=preprocess,
    output_dir=output_dir,
    config=attn_cfg,
    save_csv=True,
    csv_name=f"attention_metrics_{object_name}.csv",
    # save_summary_figures=True,
)


: 

In [ ]:
import cv2
import os
import re
from typing import Optional, Sequence


def images_to_video(
    image_folder: str,
    output_video: str,
    fps: int = 10,
    image_exts: Sequence[str] = (".png", ".jpg", ".jpeg"),
    sort_by_number: bool = True,
    resize_to_first_frame: bool = True,
    codec: Optional[str] = None,
) -> str:
    """
    将文件夹中的图片合成为视频。

    Args:
        image_folder: 图片文件夹路径
        output_video: 输出视频路径，如 "/path/to/out.avi" 或 "/path/to/out.mp4"
        fps: 帧率
        image_exts: 允许的图片后缀
        sort_by_number: 是否按文件名中的数字排序
        resize_to_first_frame: 是否将所有帧统一到第一帧尺寸
        codec: 视频编码器。
               若为 None，则根据输出后缀自动选择：
               - .avi -> XVID
               - .mp4 -> mp4v

    Returns:
        输出视频路径

    Raises:
        FileNotFoundError: 图片文件夹不存在
        ValueError: 没有图片 / 第一张图片无法读取 / VideoWriter 初始化失败
    """

    if not os.path.isdir(image_folder):
        raise FileNotFoundError(f"Image folder does not exist: {image_folder}")

    def extract_number(filename: str) -> int:
        match = re.search(r"\d+", filename)
        return int(match.group()) if match else 0

    images = [
        img for img in os.listdir(image_folder)
        if os.path.splitext(img)[1].lower() in image_exts
    ]

    if not images:
        raise ValueError(f"No images found in folder: {image_folder}")

    if sort_by_number:
        images = sorted(images, key=extract_number)
    else:
        images = sorted(images)

    print("Sorted images:", images)

    first_image_path = os.path.join(image_folder, images[0])
    first_frame = cv2.imread(first_image_path)
    if first_frame is None:
        raise ValueError(f"Cannot read first image: {first_image_path}")

    height, width = first_frame.shape[:2]

    os.makedirs(os.path.dirname(output_video), exist_ok=True)

    if codec is None:
        ext = os.path.splitext(output_video)[1].lower()
        if ext == ".avi":
            codec = "XVID"
        elif ext == ".mp4":
            codec = "mp4v"
        else:
            codec = "XVID"

    fourcc = cv2.VideoWriter_fourcc(*codec)
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    if not video.isOpened():
        raise ValueError(
            f"Failed to open VideoWriter. output={output_video}, codec={codec}"
        )

    written_count = 0

    for image_name in images:
        img_path = os.path.join(image_folder, image_name)
        frame = cv2.imread(img_path)

        if frame is None:
            print(f"Skip unreadable image: {img_path}")
            continue

        if resize_to_first_frame:
            frame = cv2.resize(frame, (width, height))

        video.write(frame)
        written_count += 1

    video.release()

    if written_count == 0:
        raise ValueError("No valid frames were written to the video.")

    print(f"Video saved: {output_video}")
    print(f"Frames written: {written_count}")
    return output_video

: 

In [ ]:
summary_folder = os.path.join(output_dir, "summary")
video_path = os.path.join(output_dir, "attention_video.mp4")

images_to_video(
    image_folder=summary_folder,
    output_video=video_path,
    fps=10,
    codec="mp4v",
)

: 

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(items[0]['images']['base_img'])
plt.show()


: 

In [ ]:
from collections import defaultdict
from typing import Any, Dict, List, Optional


def find_episode_field(sample: Dict[str, Any]) -> str:
    """
    自动猜测 episode 字段名。
    """
    candidates = [
        "episode_index",
        "episode_id",
        "episode",
        "traj_index",
        "trajectory_index",
        "sequence_index",
    ]
    for key in candidates:
        if key in sample:
            return key
    raise KeyError(
        f"Cannot find episode field in sample keys: {list(sample.keys())}"
    )


def find_step_field(sample: Dict[str, Any]) -> Optional[str]:
    """
    自动猜测 step 字段名。
    """
    candidates = [
        "frame_index",
        "step_index",
        "step",
        "timestep",
        "index_in_episode",
    ]
    for key in candidates:
        if key in sample:
            return key
    return None


def build_episode_to_indices(dataset) -> Dict[Any, List[int]]:
    """
    把 dataset 中每个 episode 对应的全局 sample index 收集起来。
    并按 step 顺序排序（如果有 step 字段）。
    """
    if len(dataset) == 0:
        return {}

    first_sample = dataset[0]
    episode_field = find_episode_field(first_sample)
    step_field = find_step_field(first_sample)

    episode_to_items = defaultdict(list)

    for global_idx in range(len(dataset)):
        sample = dataset[global_idx]
        ep_id = sample[episode_field]
        step_val = sample.get(step_field, global_idx) if step_field else global_idx
        episode_to_items[ep_id].append((step_val, global_idx))

    episode_to_indices = {}
    for ep_id, pairs in episode_to_items.items():
        pairs = sorted(pairs, key=lambda x: x[0])
        episode_to_indices[ep_id] = [idx for _, idx in pairs]

    return episode_to_indices



: 

In [ ]:

import os
import pandas as pd

def analyze_one_episode_all_layers(
    *,
    dataset,
    episode_indices,
    object_name: str,
    output_root: str,
    policy,
    preprocess,
    device,
    dataset_cfg,
    base_attn_cfg,
    layers,
    pair_mode: str = "zip",
):
    """
    对一个 episode 的所有 step，在多个 layer 上做 attention 分析。
    每个 layer 保存到独立文件夹。
    """
    os.makedirs(output_root, exist_ok=True)

    # 1) 构建该 episode 的 items（所有 step）
    items = build_analysis_items_from_dataset(
        dataset,
        indices=episode_indices,
        device=device,
        config=dataset_cfg,
        object_names=[object_name] * len(episode_indices),
        pair_mode=pair_mode,
        metadata_builder=lambda sample, idx: {
            "task_index": sample.get("task_index"),
            "episode_index": sample.get("episode_index", sample.get("episode_id")),
            "step_index": sample.get("frame_index", sample.get("step_index", idx)),
            "global_index": idx,
            "object_name": object_name,
        },
    )

    all_layer_dfs = []
    # 2) 每个 layer 单独分析
    for layer_idx in layers:
        print(f"\n===== Analyzing layer {layer_idx} =====")

        layer_output_dir = os.path.join(output_root, f"layer_{layer_idx}")
        os.makedirs(layer_output_dir, exist_ok=True)

        attn_cfg = AnalysisConfig(
            layer_idx=layer_idx,
            action_token_idx=base_attn_cfg.action_token_idx,
            head_idx=base_attn_cfg.head_idx,
            alpha=base_attn_cfg.alpha,
            top_k=base_attn_cfg.top_k,
            patch_grid_size=base_attn_cfg.patch_grid_size,
            image_configs=base_attn_cfg.image_configs,
            save_overlay_figures=base_attn_cfg.save_overlay_figures,
            save_mean_summary_figure=base_attn_cfg.save_mean_summary_figure,
            save_cluster_summary_figures=base_attn_cfg.save_cluster_summary_figures,
            add_missing_figure_paths_to_record=base_attn_cfg.add_missing_figure_paths_to_record,
        )

        df = batch_analyze_pi0_attention(
            items,
            policy=policy,
            preprocess=preprocess,
            output_dir=layer_output_dir,
            config=attn_cfg,
            save_csv=True,
            csv_name=f"attention_metrics_layer_{layer_idx}.csv",
        )

        if df is not None and len(df) > 0:
            df = df.copy()
            df["layer_idx"] = layer_idx
            all_layer_dfs.append(df)

        # 3) 把该 layer 的 summary 图做成视频
        summary_folder = os.path.join(layer_output_dir, "summary")
        video_path = os.path.join(layer_output_dir, f"attention_video_layer_{layer_idx}.mp4")

        if os.path.isdir(summary_folder):
            pngs = [f for f in os.listdir(summary_folder) if f.lower().endswith(".png")]
            if len(pngs) > 0:
                images_to_video(
                    image_folder=summary_folder,
                    output_video=video_path,
                    fps=10,
                    codec="mp4v",
                )
                print(f"Saved video: {video_path}")
            else:
                print(f"No PNG files found in {summary_folder}, skip video.")
        else:
            print(f"Summary folder not found: {summary_folder}")

    # 4) 合并所有 layer 的 df
    merged_csv_path = None
    if len(all_layer_dfs) > 0:
        merged_df = pd.concat(all_layer_dfs, ignore_index=True)
        merged_csv_path = os.path.join(output_root, "attention_metrics_all_layers.csv")
        merged_df.to_csv(merged_csv_path, index=False)
        print(f"Saved merged CSV: {merged_csv_path}")
    else:
        merged_df = None

    return merged_df, merged_csv_path

: 

In [ ]:
import os
from collections import Counter
from datasets import load_dataset


def get_object_names_for_indices(dataset, indices, task_to_object):
    objs = []
    for idx in indices:
        sample = dataset[idx]
        task_id = sample.get("task_index")
        obj = task_to_object.get(task_id, "unknown")
        objs.append(obj)
    return objs


def get_main_object_for_episode(dataset, episode_indices, task_to_object):
    """
    对一个 episode，统计其中所有 sample 的 task_index，
    取出现最多的那个 task 对应的 object_name。
    """
    task_ids = []
    for idx in episode_indices:
        sample = dataset[idx]
        task_id = sample.get("task_index")
        task_ids.append(task_id)

    if len(task_ids) == 0:
        return "unknown"

    main_task_id = Counter(task_ids).most_common(1)[0][0]
    return task_to_object.get(main_task_id, "unknown")


def analyze_multiple_episodes(
    *,
    dataset,
    episode_ids,
    episode_to_indices,
    task_to_object,
    output_root,
    policy,
    preprocess,
    device,
    dataset_cfg,
    base_attn_cfg,
    layers,
):
    results = {}

    for ep_id in episode_ids:
        print("\n\n############################")
        print(f"Processing episode: {ep_id}")
        print("############################")

        episode_indices = episode_to_indices[ep_id]

        # 自动确定这个 episode 对应的 object
        object_name = get_main_object_for_episode(
            dataset,
            episode_indices,
            task_to_object,
        )
        print(f"Detected object for episode {ep_id}: {object_name}")

        ep_output_root = os.path.join(output_root, f"episode_{ep_id}_{object_name}")

        merged_df, merged_csv_path = analyze_one_episode_all_layers(
            dataset=dataset,
            episode_indices=episode_indices,
            object_name=object_name,
            output_root=ep_output_root,
            policy=policy,
            preprocess=preprocess,
            device=device,
            dataset_cfg=dataset_cfg,
            base_attn_cfg=base_attn_cfg,
            layers=layers,
            pair_mode="zip",
        )

        results[ep_id] = {
            "object_name": object_name,
            "episode_indices": episode_indices,
            "merged_df": merged_df,
            "merged_csv_path": merged_csv_path,
        }

    return results

: 

In [ ]:
ds = load_dataset(
    "lerobot/libero_object_image",
    cache_dir="../mydatasets/amylingchen/libero"
)
train_ds = ds["train"]
train_ds = train_ds.select(range(4000))

TASK_TO_OBJECT = {
    0: "orange juice",
    1: "ketchup",
    2: "cream cheese",
    3: "bbq sauce",
    4: "alphabet soup",
    5: "milk",
    6: "salad dressing",
    7: "butter",
    8: "tomato sauce",
    9: "chocolate pudding",
}

dataset_cfg = DatasetBuildConfig(
    resize_size=(224, 224),
    image_key="observation.images.image",
    wrist_image_key="observation.images.wrist_image",
    output_base_image_key="observation.images.image",
    output_wrist_image_key="observation.images.image2",
    state_key="observation.state",
    instruction_template="pick up the {object} and place it in the basket",
)

base_attn_cfg = AnalysisConfig(
    layer_idx=-1,   # 会在 layer 循环中被覆盖
    action_token_idx=-1,
    head_idx=None,
    alpha=0.35,
    top_k=5,
    save_overlay_figures=False,
    save_mean_summary_figure=True,
    save_cluster_summary_figures=False,
)

: 

In [ ]:
episode_to_indices = build_episode_to_indices(train_ds)

# 先看前几个 episode
episode_ids = list(episode_to_indices.keys())[:1]
layers =list(range(-1,-2,-1))  # 分析最后 12 层
results = analyze_multiple_episodes(
    dataset=train_ds,
    episode_ids=episode_ids,
    episode_to_indices=episode_to_indices,
    task_to_object=TASK_TO_OBJECT,
    output_root="../output/pi0_attention_multi_episode",
    policy=policy,
    preprocess=preprocess,
    device=device,
    dataset_cfg=dataset_cfg,
    base_attn_cfg=base_attn_cfg,
    layers=layers,
)

: 

In [ ]:
train_ds

: 

: 